In [1]:
import polars as pl
pl.Config.set_tbl_rows(30)

polars.config.Config

In [2]:
df = pl.read_csv("results.csv")

In [8]:
(
    df
    .filter(pl.col("split").eq("val"))
    .group_by("method", "model_version", "reasoning", "effort")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean(),
        pl.len().alias("n")
    )
    .sort("correct_board_mean", descending=True)
    .filter(pl.col("method").eq("cnn"))
)

method,model_version,reasoning,effort,correct_square_mean,correct_board_mean,n
str,str,str,str,f64,f64,u32
"""cnn""","""optimised_plus_prior_correctio…","""none""","""high""",0.739553,0.947831,5
"""cnn""","""optimised""","""none""","""high""",0.727543,0.910303,5
"""cnn""","""optimised_10k""","""none""","""high""",0.708313,0.909649,5
"""cnn""","""square_global""","""none""","""high""",0.716104,0.859952,5
"""cnn""","""square_per_square""","""none""","""high""",0.759768,0.858538,5
"""cnn""","""optimised_5k""","""none""","""high""",0.63859,0.788188,5
"""cnn""","""none_global""","""none""","""high""",0.602451,0.307201,5


In [42]:
(
    df.filter(
        pl.col("model_version").str.starts_with("optimised"), 
        ~pl.col("model_version").str.contains("prior"),
        pl.col("split").eq("val")
    )
    .group_by("method", "model_version")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean()
    )
    .select(
        pl.col("method").replace_strict({"cnn": "CNN"}).alias("model type"),
        pl.col("model_version").replace_strict({"optimised": 13184, "optimised_10k": 9538, "optimised_5k": 4864}).alias("n"),
        pl.col("correct_square_mean").round(3).alias("square accuracy"),
        pl.col("correct_board_mean").round(3).alias("move accuracy")
    )
)

model type,n,square accuracy,move accuracy
str,i64,f64,f64
"""CNN""",13184,0.728,0.91
"""CNN""",4864,0.639,0.788
"""CNN""",9538,0.708,0.91


In [32]:
model_version_map = {
    "claude-opus-5": "Opus 5",
    "claude-sonnet-5": "Sonnet 5"
}

reasoning_enabled_map = {
    "thinking": "yes",
    "none": "no"
}




(
    df
    .filter(pl.col("split").eq("val"))
    .group_by("method", "model_version", "reasoning", "effort")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean(),
        pl.len().alias("n")
    )
    .sort("correct_board_mean", descending=True)
    .group_by("method", "model_version", "reasoning")
    .agg(
        pl.col("correct_square_mean").first(),
        pl.col("correct_board_mean").first(),
        pl.col("effort").first()
    )
    .filter(~pl.col("method").is_in(["cnn", "fen_whole"]), pl.col("model_version").ne("claude-opus-4-8"))
    .select(
        pl.col("model_version").replace_strict(model_version_map).alias("model version"),
        pl.col("method").str.starts_with("square").replace_strict({True: "yes", False: "no"}).alias("classify squares first?"),
        pl.col("method").alias("prompt version"),
        pl.when(pl.col("reasoning").eq("thinking"))
        .then(pl.col("effort").replace_strict({"low": "yes (low effort)", "medium": "yes (medium effort)", "high": "yes (high effort)"}))
        .otherwise(pl.lit("no"))
        .alias("reasoning enabled?"),
        pl.col("correct_board_mean").alias("move accuracy"),
        pl.col("correct_square_mean").alias("square accuracy")
    )
    .with_columns(
        pl.when(pl.col("classify squares first?").eq("no"))
        .then(pl.lit(None))
        .otherwise(pl.col("square accuracy"))
        .alias("square accuracy")
    )
).write_csv("claude_results.csv")

# How to turn this into an actual table I can show in the blog post?
# Columns need to be renamed 
    # reasoning -> Reasoning enabled?
    # correct_square_mean -> Square Accuracy
    # correct_board_mean -> Move Accuracy
# Values need to be renamed:
    # claude-opus-5 -> Opus 5
    # correct

In [25]:
df.filter(pl.col("model_version").eq("claude-opus-5"), pl.col("split").eq("test"))

method,model_version,prompt_version,reasoning,prior_correction,data_path,split,setup_id,n_boards,correct_square,correct_square_mean,correct_board,correct_board_mean,board_rank,board_rank_mean,first_output_illegal,first_output_illegal_mean,none_legal,none_legal_mean,input_tokens,input_tokens_mean,output_tokens,output_tokens_mean,inference_time,inference_time_mean,cost,cost_mean,n_suggested,n_suggested_mean,effort
str,str,i64,str,bool,str,str,str,i64,str,f64,str,f64,str,f64,str,str,str,str,str,f64,str,f64,str,f64,str,f64,str,str,str
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-01_210216""",7,"""[0.484375, 0.4375, 0.46875, 0.…",0.448661,"""[true, true, false, false, fal…",0.285714,"""[1.0, 1.0, 0.43333333333333335…",0.874044,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55577, 55577, 55577, 55577, 5…",55577.0,"""[5028, 5080, 5099, 5161, 5084,…",5099.857143,"""[null, null, null, null, null,…",null,"""[0.20179250000000004, 0.202442…",0.202691,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-06_204708""",9,"""[0.421875, 0.5, 0.5625, 0.4062…",0.458333,"""[false, false, false, true, fa…",0.111111,"""[0.5588235294117647, 0.875, 0.…",0.873473,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[54902, 54902, 54902, 54902, 5…",54902.0,"""[4912, 5222, 5002, 5208, 5155,…",5240.111111,"""[null, null, null, null, null,…",null,"""[0.19865500000000003, 0.202530…",0.202756,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_083055""",8,"""[0.578125, 0.5625, 0.515625, 0…",0.542969,"""[true, false, false, false, fa…",0.375,"""[1.0, 0.9411764705882353, 0.97…",0.910561,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55013, 55013, 55013, 55013, 5…",55013.0,"""[4957, 4969, 5166, 5042, 4967,…",4987.5,"""[null, null, null, null, null,…",null,"""[0.199495, 0.19964500000000002…",0.199876,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_090013""",7,"""[0.578125, 0.4375, 0.4375, 0.4…",0.439732,"""[true, false, false, false, fa…",0.142857,"""[1.0, 0.875, 0.928571428571428…",0.856468,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[54939, 54939, 54939, 54939, 5…",54939.0,"""[5135, 5301, 5092, 5640, 5122,…",5272.0,"""[null, null, null, null, null,…",null,"""[0.20153500000000002, 0.20361,…",0.2032475,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_090457""",6,"""[0.390625, 0.453125, 0.5, 0.48…",0.46875,"""[false, false, false, false, f…",0.0,"""[0.972972972972973, 0.75, 0.53…",0.769463,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55287, 55287, 55287, 55287, 5…",55287.0,"""[5626, 5039, 5256, 5186, 4907,…",5185.666667,"""[null, null, null, null, null,…",null,"""[0.20854250000000002, 0.201205…",0.203038,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_090950""",7,"""[0.546875, 0.46875, 0.53125, 0…",0.495536,"""[false, false, false, false, f…",0.0,"""[0.8409090909090909, 0.9555555…",0.791897,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55599, 55599, 55599, 55599, 5…",55599.0,"""[4864, 4959, 5068, 5140, 4914,…",5009.571429,"""[null, null, null, null, null,…",null,"""[0.19979750000000002, 0.200985…",0.201617,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-

In [23]:
import os
os.getcwd()

'c:\\Users\\User\\Documents\\Coding\\robot-chess-commentator\\evaluation'

In [20]:
df.filter(pl.col("model_version").eq("claude-sonnet-5"), pl.col("method").eq("board")).select("reasoning", "correct_square", "correct_square_mean", "correct_board", "correct_board_mean", "board_rank", "n_suggested", "none_legal", "first_output_illegal", "n_suggested_mean"),

(shape: (10, 10)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ reasoning ┆ correct_s ┆ correct_s ┆ correct_b ┆ … ┆ n_suggest ┆ none_lega ┆ first_out ┆ n_sugges │
 │ ---       ┆ quare     ┆ quare_mea ┆ oard      ┆   ┆ ed        ┆ l         ┆ put_illeg ┆ ted_mean │
 │ str       ┆ ---       ┆ n         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ al        ┆ ---      │
 │           ┆ str       ┆ ---       ┆ str       ┆   ┆ str       ┆ str       ┆ ---       ┆ str      │
 │           ┆           ┆ f64       ┆           ┆   ┆           ┆           ┆ str       ┆          │
 ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ none      ┆ [0.9375,  ┆ 0.944444  ┆ [false,   ┆ … ┆ [1, 1, 1, ┆ [false,   ┆ [false,   ┆ 1.222222 │
 │           ┆ 0.953125, ┆           ┆ false,    ┆   ┆ 1, 1, 1,  ┆ true,     ┆ true,     ┆ 22222222 │
 │           ┆ 0.9375,   ┆           ┆ false,    ┆   ┆ 1, 1, 1,  

In [5]:
df.columns

['method',
 'model_version',
 'prompt_version',
 'reasoning',
 'prior_correction',
 'data_path',
 'split',
 'setup_id',
 'n_boards',
 'correct_square',
 'correct_square_mean',
 'correct_board',
 'correct_board_mean',
 'board_rank',
 'board_rank_mean',
 'first_output_illegal',
 'first_output_illegal_mean',
 'none_legal',
 'none_legal_mean',
 'input_tokens',
 'input_tokens_mean',
 'output_tokens',
 'output_tokens_mean',
 'inference_time',
 'inference_time_mean',
 'cost',
 'cost_mean',
 'n_suggested',
 'n_suggested_mean',
 'effort']

In [5]:
df["method"].unique()

method
str
"""square_label"""
"""square_logits"""
"""fen_whole"""
"""move"""
"""board"""
"""cnn"""


In [6]:
(
    df.filter(pl.col("split").eq("test"), pl.col("model_version").str.contains("claude"))
    .group_by("method", "model_version", "reasoning", "effort")
    .agg(
        pl.col("correct_board_mean").mean()
    )
    .sort("correct_board_mean", descending=True)
)

method,model_version,reasoning,effort,correct_board_mean
str,str,str,str,f64
"""square_logits""","""claude-opus-5""","""thinking""","""low""",0.226659


In [16]:
(
    df.filter(pl.col("method").eq("square_logits"))
    .select("n_boards", "correct_square_mean", "correct_board_mean", "board_rank", "model_version")
)

n_boards,correct_square_mean,correct_board_mean,board_rank,model_version
i64,f64,f64,str,str
18,0.228299,0.0,"""[0.6666666666666667, 0.8, 0.71…","""claude-sonnet-5"""
18,0.215278,0.111111,"""[0.7142857142857143, 0.65, 1.0…","""claude-sonnet-5"""
17,0.205882,0.0,"""[0.8666666666666667, 0.6428571…","""claude-sonnet-5"""
11,0.268466,0.090909,"""[0.32352941176470584, 0.555555…","""claude-sonnet-5"""
10,0.2515625,0.1,"""[0.45238095238095233, 0.354838…","""claude-sonnet-5"""
18,0.159722,0.333333,"""[0.5384615384615384, 1.0, null…","""claude-sonnet-5"""
17,0.215074,0.058824,"""[0.9333333333333333, 1.0, 0.71…","""claude-sonnet-5"""
11,0.296875,0.090909,"""[0.4117647058823529, 0.3888888…","""claude-sonnet-5"""
10,0.2671875,0.1,"""[0.26190476190476186, 0.451612…","""claude-sonnet-5"""


In [9]:
df.filter(pl.col("method").eq("move"))

method,model_version,prompt_version,reasoning,prior_correction,data_path,split,setup_id,n_boards,correct_square,correct_square_mean,correct_board,correct_board_mean,board_rank,board_rank_mean,first_output_illegal,first_output_illegal_mean,none_legal,none_legal_mean,input_tokens,input_tokens_mean,output_tokens,output_tokens_mean,inference_time,inference_time_mean,cost,cost_mean
str,str,i64,str,bool,str,str,str,i64,str,f64,str,f64,str,f64,str,str,str,str,str,f64,str,f64,str,f64,str,f64
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-09_205328""",18,"""[null, null, null, null, null,…",null,"""[true, true, false, false, fal…",0.111111,"""[null, 1.0, null, null, null, …",0.625,"""[true, false, false, true, fal…","""0.4444444444444444""","""[false, false, false, false, f…","""0.2777777777777778""","""[998, 996, 997, 998, 1000, 100…",1003.444444,"""[19, 24, 34, 27, 39, 29, 45, 4…",48.111111,"""[4.125, 2.5470000000000255, 2.…",4.164111,"""[0.005465, 0.00558, 0.00583500…",0.00622
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_200019""",17,"""[null, null, null, null, null,…",null,"""[true, false, false, false, fa…",0.058824,"""[1.0, null, null, null, null, …",1.0,"""[false, true, false, true, tru…","""0.5294117647058824""","""[false, true, false, false, tr…","""0.47058823529411764""","""[1047, 1050, 1054, 1056, 1057,…",1056.882353,"""[59, 59, 59, 59, 54, 49, 59, 5…",60.823529,"""[4.234000000000151, 3.25, 2.90…",4.151706,"""[0.00671, 0.006725, 0.006745, …",0.006805
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_201154""",11,"""[null, null, null, null, null,…",null,"""[false, false, false, false, f…",0.0,"""[null, 0.33333333333333337, nu…",0.333333,"""[false, false, false, false, t…","""0.45454545454545453""","""[false, false, false, false, f…","""0.18181818181818182""","""[1012, 1012, 1010, 1010, 1011,…",1009.090909,"""[49, 49, 59, 49, 59, 59, 59, 5…",55.727273,"""[2.7029999999999745, 3.0309999…",3.957273,"""[0.006285000000000001, 0.00628…",0.006439
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_203121""",10,"""[null, null, null, null, null,…",null,"""[false, false, false, false, f…",0.0,"""[null, null, null, null, null,…",null,"""[true, false, false, false, tr…","""0.5""","""[true, false, false, false, tr…","""0.5""","""[1022, 1020, 1019, 1020, 1020,…",1019.8,"""[59, 59, 64, 104, 74, 59, 84, …",72.5,"""[3.1569999999999254, 3.75, 6.6…",3.9267,"""[0.006585000000000001, 0.00657…",0.006912
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_203700""",18,"""[null, null, null, null, null,…",null,"""[false, false, false, false, t…",0.055556,"""[null, null, null, null, 1.0, …",1.0,"""[false, true, true, false, fal…","""0.5555555555555556""","""[false, true, true, false, fal…","""0.5555555555555556""","""[999, 999, 998, 998, 1000, 997…",996.555556,"""[59, 64, 54, 39, 69, 54, 49, 5…",60.888889,"""[2.969000000000051, 3.75, 3.85…",3.633667,"""[0.00647, 0.006595, 0.00634, 0…",0.006505


In [8]:
df.group_by(["method", "model_version"]).agg(pl.col("correct_board_mean").mean(), pl.col("none_legal").mean(), pl.len().alias("n_setups"))

method,model_version,correct_board_mean,none_legal,n_setups
str,str,f64,str,u32
"""cnn""","""none_global""",0.478962,null,16
"""cnn""","""optimised_5k""",0.691373,null,16
"""cnn""","""optimised_plus_prior_correctio…",0.946495,null,16
"""cnn""","""square_per_square""",0.894905,null,16
"""cnn""","""optimised_10k""",0.88062,null,16
"""move""","""claude-opus-4-8""",0.045098,null,5
"""cnn""","""square_global""",0.913824,null,16
"""cnn""","""optimised""",0.945928,null,16


In [3]:
import os
os.getcwd()

'c:\\Users\\User\\Documents\\Coding\\robot-chess-commentator\\evaluation'